# ⚠️ Optional/heavy: a LoRA fine-tune you can read

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 48 §48.2–§48.5 · type: walkthrough*

*One-line promise:* read and run a *minimal* LoRA adapter fit on a tiny model — see trainable params drop under 1% — and place DPO, distillation, and the agent-training ladder on the conceptual cost ladder. **The default run is fully mocked: no GPU, no key, no network.**

## 🧠 Why this matters

LoRA is a *commodity*: freeze the base model, learn small low-rank adapter matrices, train well under 1% of the parameters, ship the adapter as a tiny swappable file. Knowing the API is not the skill. The skill is knowing that **data curation and a pre-defined eval are the real work**, that an adapter changes *form, not facts*, and that everything above LoRA — DPO, distillation, agentic RL — is a **cost ladder** you descend only when the cheaper rung plateaus. You'll mostly *commission or evaluate* these, so you need to know what each buys and what it costs.

## Objectives & prereqs

**By the end you can:**
- Explain what LoRA freezes and what it trains, and why `print_trainable_parameters()` shows <1%.
- Run a *toy* adapter fit (mocked by default) with a **held-out set defined before training**.
- Predict and confirm that an adapter enforces a **format** but cannot **recall a new fact**.
- Locate **DPO**, **distillation**, and the **agent-training ladder** (tool-use FT → PRM → agentic RL → synthetic trajectories) on the cost ladder.

**Prereqs:** `48-01` (triage) · Ch 22 (define the eval/held-out set *before* training) · Ch 39 (serving adapters on a shared base, rollback) — referenced, not required to run.

**Cost:** `MOCK=1` (default) is free, offline, GPU-free. `MOCK=0` actually trains and needs a GPU + extra deps — declared in the gate cell below and **skipped in CI**.

## ⚠️ Gate: this notebook is optional and heavy

The default path loads a **canned training log + adapter stats** so the notebook runs green for everyone. The live path (`MOCK=0`) trains a real toy adapter and needs:

- a GPU (even a tiny base is slow on CPU),
- extra packages **not** in the base `requirements.txt`: `peft`, `transformers`, `datasets`, `accelerate`, `torch`,
- a small local base model download (e.g. `Qwen/Qwen2.5-0.5B`).

Leave `COMPANION_MOCK=1` unless you specifically want to train. The mock is faithful to the *shapes* you'd see live.

In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # git-ignored .env if present; never hardcode keys

# MOCK=1 (default): load canned train log + adapter stats. Runs FREE, OFFLINE,
# NO GPU, NO KEY. MOCK=0: actually train (needs GPU + the extra deps above).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Tiny local base used ONLY on the live path. Small on purpose so a curious
# reader with a GPU can finish in minutes; never downloaded in MOCK mode.
BASE_MODEL = os.getenv("COMPANION_BASE_MODEL", "Qwen/Qwen2.5-0.5B")

random.seed(7)  # any mock jitter is reproducible

DATA = Path("data")

print("MOCK =", MOCK, "| base (live only) =", BASE_MODEL)
if not MOCK:
    print("\n⚠️  MOCK=0: this will train a real adapter and needs a GPU + peft/transformers/datasets/accelerate/torch.")

## Step 1 — curate the data, and hold out a test set *first*

The unglamorous truth: the technique is a commodity; **data curation wins fine-tunes**. A few thousand clean, deduplicated, correctly-formatted examples that actually exhibit the target behavior beat a million scraped ones. Our toy task is pure *form*: turn a free-text order note into a strict JSON status. We split a held-out set **before** training — the eval is defined first, or you can't tell if training helped.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


train = read_jsonl(DATA / "toy_train.jsonl")
holdout = read_jsonl(DATA / "toy_holdout.jsonl")

print(f"{len(train)} train examples, {len(holdout)} holdout (defined BEFORE training)")
print("\nexample train pair:")
print("  in :", train[0]["input"])
print("  out:", train[0]["output"])
print("\nholdout deliberately mixes two probes:")
for h in holdout:
    print(f"  {h['id']} tests={h['tests']:<7} {h['input'][:50]}")

## Step 2 — configure LoRA (freeze base, train tiny adapters)

This is the code shape from §48.2: pick a low rank `r`, a handful of `target_modules` on the attention projections, and wrap the frozen base with `get_peft_model`. `print_trainable_parameters()` reveals the whole point — you're training a sliver. In `MOCK=1` we load the captured numbers from `data/adapter_stats.json` instead of building a real model.

In [ ]:
LORA_CONFIG = {
    "r": 8,                       # the rank — the main capacity/size dial
    "lora_alpha": 16,             # scaling; commonly ~2x r
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj"],  # which projections get adapters
    "task_type": "CAUSAL_LM",
}


def configure_lora():
    """Return (trainable, total, percent). MOCK: read captured stats; live: build it."""
    if MOCK:
        stats = json.loads((DATA / "adapter_stats.json").read_text(encoding="utf-8"))
        return stats["trainable_params"], stats["base_total_params"], stats["trainable_percent"]
    # --- live path (MOCK=0): the real §48.2 shape ---
    from peft import LoraConfig, get_peft_model       # noqa: F401  (extra dep)
    from transformers import AutoModelForCausalLM      # noqa: F401  (extra dep)

    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
    config = LoraConfig(**LORA_CONFIG)
    model = get_peft_model(base, config)
    model.print_trainable_parameters()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, trainable / total


trainable, total, pct = configure_lora()
print(f"trainable params: {trainable:,}")
print(f"total params:     {total:,}")
print(f"trainable %:      {pct:.4%}   <-- the entire point of PEFT")

Under **1%** of the parameters are trained. The frozen base is shared; the adapter is a small file you can swap per tenant or task and roll back instantly (Ch 39). QLoRA pushes further — quantize the frozen base to 4-bit and a 7–13B fine-tune fits on one GPU.

## Step 3 — "train" and read the loss curve

On the live path you'd run a `Trainer` for a few epochs. In `MOCK=1` we load `data/train_log.json` — a captured loss curve — so you can read the *shape* (loss falling, then flattening) without a GPU. A falling-then-flat curve on a tiny set is exactly what format-shaping looks like: the model is learning the *habit*, fast.

In [ ]:
def get_train_log():
    if MOCK:
        return json.loads((DATA / "train_log.json").read_text(encoding="utf-8"))
    # --- live path would assemble + run a Trainer and return its history ---
    raise NotImplementedError(
        "Live training loop is left as the opt-in heavy path; wire up "
        "transformers.Trainer over the train split and capture .state.log_history."
    )


log = get_train_log()
print(f"run: {log['run_id']}  base: {log['base_model']}  method: {log['method']}")
print(f"epochs: {log['config']['epochs']}  r: {log['config']['r']}  "
      f"targets: {log['config']['target_modules']}\n")
print("loss curve:")
for s in log["steps"]:
    bar = "█" * round(s["loss"] * 8)
    print(f"  epoch {s['epoch']:.2f}  loss {s['loss']:.2f}  {bar}")
print(f"\nwall clock: {log['wall_clock_seconds']}s on {log['device']}")

## 🔮 Predict

Two probes wait in the holdout set. Before you run the eval, predict each:

1. The **format probe** (`tests=format`): a new order note — will the adapter emit the strict JSON shape it was trained on?
2. The **fact probe** (`tests=fact`): "what is the capital of the country where this order was delivered?" — a fact never in the training data.

Write **yes/no** for each. The chapter's whole point about *form vs facts* is riding on your answer. Now evaluate.

In [ ]:
def adapter_infer(example: dict) -> str:
    """What the LoRA-tuned model returns. MOCK: a faithful stand-in of the behavior.

    The adapter reliably reproduces the *trained format* but cannot conjure a
    *new fact* it never saw — the core form-vs-facts lesson, made observable.
    """
    if not MOCK:
        # live: tokenize prompt, model.generate(...), decode. (heavy path)
        raise NotImplementedError("Run generation on the merged/adapter model here.")
    text = example["input"].lower()
    # FORMAT habit: the adapter learned to map notes -> strict JSON status.
    import re
    m = re.search(r"order (\d+)", text)
    oid = int(m.group(1)) if m else None
    if "deliver" in text or "handed to the customer" in text or "arrived at the customer" in text:
        status, eta = "delivered", None
    elif "pack" in text or "queue" in text or "process" in text:
        status, eta = "processing", None
    elif "ship" in text or "out for delivery" in text or "depot" in text:
        status = "shipped"
        d = re.search(r"(\d{4}-\d{2}-\d{2})", example["input"])
        eta = d.group(1) if d else None
    else:
        status, eta = "unknown", None
    # The FACT probe asks for world knowledge never in the data: the tuned model
    # does NOT gain it. A format-tuned model will dutifully emit its trained shape
    # and simply has no field for the fact — it cannot recall what it never learned.
    if "capital" in text:
        return json.dumps({"order_id": oid, "status": status, "eta": eta})  # no 'capital' — fact absent
    return json.dumps({"order_id": oid, "status": status, "eta": eta})


def eval_holdout():
    fmt_ok = fmt_total = fact_handled = fact_total = 0
    for h in holdout:
        out = adapter_infer(h)
        if h["tests"] == "format":
            fmt_total += 1
            try:
                obj = json.loads(out)
                ok = set(obj.keys()) == {"order_id", "status", "eta"}
            except json.JSONDecodeError:
                ok = False
            fmt_ok += ok
            print(f"  {h['id']} format  {'PASS' if ok else 'FAIL'}  -> {out}")
        else:
            fact_total += 1
            # The 'right' behavior is to NOT invent the fact. Our adapter emits the
            # trained shape with no fabricated capital -> it did not gain the fact.
            invented_fact = "capital" in out.lower()
            fact_handled += (not invented_fact)
            print(f"  {h['id']} fact    {'did NOT invent (good)' if not invented_fact else 'HALLUCINATED'}  -> {out}")
    print(f"\nformat adherence: {fmt_ok}/{fmt_total}")
    print(f"fact probes that correctly gained-no-new-fact: {fact_handled}/{fact_total}")


eval_holdout()

Confirmed: the adapter **enforces the format** (every format probe passes) and **does not recall a new fact** (it never invents a capital it was never taught). This is §48.2 made observable — *fine-tune for form and behavior; retrieve for facts.* If you needed that capital, the fix is RAG, not more epochs.

## ⚠️ Pitfall: a dirty, duplicated, under-curated dataset

The fastest way to waste a fine-tune is bad data. A few thousand *clean* examples beat a million scraped ones — and duplicates, label noise, leaked PII, and licensing problems all quietly poison the result. Here's a tiny curation pass that catches the usual suspects before they reach the trainer.

In [ ]:
def audit_dataset(rows):
    seen, dups, empties, pii = set(), [], [], []
    import re
    email = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
    for r in rows:
        key = (r["input"].strip().lower(), r["output"].strip())
        if key in seen:
            dups.append(r["id"])
        seen.add(key)
        if not r["input"].strip() or not r["output"].strip():
            empties.append(r["id"])
        if email.search(r["input"]) or email.search(r["output"]):
            pii.append(r["id"])
    return {"duplicates": dups, "empties": empties, "possible_pii": pii}


# Inject a couple of bad rows to show the audit firing (NOT added to training).
dirty = train + [
    {"id": "bad-dup", "instruction": train[0]["instruction"], "input": train[0]["input"], "output": train[0]["output"]},
    {"id": "bad-pii", "instruction": "x", "input": "Order 1 for jane@acme.com shipped", "output": "{}"},
]
report = audit_dataset(dirty)
for k, v in report.items():
    print(f"{k:>14}: {v}")
print("\nVersion the *clean* set, and define the eval before you collect a single example.")

## 🧠 The cost ladder beyond LoRA (conceptual — nothing trains here)

LoRA tunes single-turn *behavior*. Agents add **multi-step trajectories** — sequences of tool calls and decisions — and a rougher toolkit grew up around improving that whole loop. You'll mostly *buy, commission, or decline* these, so the job is knowing what each rung buys and what it costs. Read this as a **cost ladder**: descend only when the cheaper rung demonstrably plateaus on your eval.

In [ ]:
COST_LADDER = [
    {
        "technique": "Tool-use fine-tuning",
        "buys": "Reliable, correctly-formatted tool calls from a smaller, cheaper model",
        "when_cost": "Stable tool set, proven task; cheap — the standard cost-down move",
    },
    {
        "technique": "Process reward model (PRM)",
        "buys": "Step-level credit so long-horizon training actually converges",
        "when_cost": "Long multi-step tasks; needs per-step labels or a learned grader",
    },
    {
        "technique": "Agentic RL on rollouts",
        "buys": "Genuine multi-step competence beyond single-turn tuning",
        "when_cost": "Highest cost: rollout environment, infra, and a reward you trust",
    },
    {
        "technique": "Synthetic trajectories",
        "buys": "Distill a strong agent's verified successes into a small one",
        "when_cost": "A good verifier exists; just filtered fine-tuning — much cheaper than full RL",
    },
]

print("LEVERS FOR IMPROVING AGENTS (read top->bottom as a cost ladder)\n")
for rung in COST_LADDER:
    print(f"• {rung['technique']}")
    print(f"    buys : {rung['buys']}")
    print(f"    cost : {rung['when_cost']}\n")

### DPO and distillation, in one breath each

- **DPO (direct preference optimization)** reaches an RLHF-like result *without* the RL machinery: train directly on **(prompt, preferred, rejected)** triples with a simple loss. It fits cases where quality is easy to *judge* but hard to *specify* (tone, helpfulness, fuzzy policy), and comparisons are far cheaper to collect than gold outputs.
- **Distillation** uses a strong *teacher* to teach a small *student*: generate or grade data with the teacher, fine-tune the student on it. It's the economic engine — prototype on a frontier model, then distill the proven behavior into something **10–50× cheaper** to serve at volume.

In [ ]:
# A DPO training row is just a triple. No RL, no reward model — a pair the model
# learns to prefer. (Shape only; we are not training.)
dpo_example = {
    "prompt": "Customer: my package is late and I'm furious.",
    "preferred": "I'm sorry it's late — here's exactly where it is and what I'll do now: ...",
    "rejected": "Per policy section 4.2, delays are not guaranteed against.",
}
print("a DPO triple (what you'd collect, by the thousand):")
print(json.dumps(dpo_example, indent=2))
print("\nNote: pairwise 'A is better than B' is far cheaper to label than a perfect gold answer.")

### 🔑 The signal under all of it (the chapter's key idea)

Every rung depends on one thing: **a signal that says the trajectory was good.** That signal *is* your eval suite from Ch 22 — mechanically, the same harness is the reward function and the rejection-sampling verifier. **An agent you cannot evaluate is an agent you cannot train.** And whatever you train still has to be *served*: a tuned agent model is an adapter or a deployment on the same serving plane (Ch 39), with the same rollback discipline.

## 🎯 Senior lens

Every fine-tune is an asset that **depreciates** — the next base model may match it out of the box, and your adapter doesn't transfer. So keep the prompt fallback alive so a base-model swap is an *afternoon, not a quarter*; treat the **curated dataset and the eval** as the durable product and the weights as a disposable artifact; and descend the cost ladder one rung at a time, only when a measured plateau forces you. The team that owns the data and the eval owns the capability — whoever owns this quarter's checkpoint owns a liability with a half-life.

## Recap

- **LoRA = freeze the base, train low-rank adapters (<1% of params)**; the adapter is a tiny swappable, roll-back-able file. QLoRA quantizes the frozen base to fit bigger models on one GPU.
- **Data curation + a pre-defined held-out eval are the real work**; the technique is a commodity. Define the eval *before* collecting data.
- An adapter changes **form, not facts** — it enforces a format but cannot recall new knowledge (that's RAG's job).
- **DPO** trains on preference triples without RL; **distillation** compresses a teacher into a 10–50× cheaper student.
- The **agent cost ladder** (tool-use FT → PRM → agentic RL → synthetic trajectories) all rests on a trusted eval signal. No eval, no training.

## Exercises

1. **Change the rank.** In `LORA_CONFIG` set `r` to 4 and to 32. 🔮 Predict how trainable-% moves *before* editing `data/adapter_stats.json` to match, then update it and re-run — does the file's `trainable_percent` agree with your intuition about rank vs adapter size?
2. **Draft a DPO triple.** Write one `(prompt, preferred, rejected)` triple for *your* product's tone. What makes the preferred answer better in a way you could *judge* but struggle to *specify*?
3. **Add a holdout probe.** Append one `tests=fact` row to `data/toy_holdout.jsonl` and confirm the eval still reports the adapter did not invent the fact. Why is "refused to invent" the success condition here?
4. **Place a real task on the ladder.** Pick something at your job and name which rung (tool-use FT / PRM / agentic RL / synthetic trajectories) you'd reach for — and the cheaper rung you'd prove plateaued first.

In [ ]:
# Exercise 1 — set r in {4, 32}; predict trainable-%, then reconcile with adapter_stats.json.


In [ ]:
# Exercise 2 — draft a (prompt, preferred, rejected) DPO triple for your tone.


In [ ]:
# Exercise 3 — add a fact probe to data/toy_holdout.jsonl and re-run eval_holdout().


## Next

- **Back to the decision:** [`48-01-customization-triage.ipynb`](./48-01-customization-triage.ipynb) — the triage that should gate ever reaching this notebook.
- **The eval that gates every technique here:** [`blueprints/eval-harness/`](../../../blueprints/eval-harness/) — your eval suite *is* the reward function and the rejection-sampling verifier.
- **Serving + rollback for any custom model/adapter:** the `capstone/` serving plane (Ch 39) — a fine-tuned model is "an adapter or a deployment" with the same rollback discipline as any release.
- **Next chapter:** [`../49-frontier-and-staying-current/`](../49-frontier-and-staying-current/) — keeping all of this current as base models ship.